# 02 模型结构 / 订正前后 / 中间表示
用随机或真实 batch 跑一遍模型，检查各 shape、订正效果、门控权重。

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))
import torch, numpy as np
from src.utils.config import load_config
from src.trainers.build import build_station_data, build_models
cfg = load_config('../configs/default.yaml')
# 切到 code/ 以匹配相对路径
os.chdir(os.path.abspath('..'))
SID='station00'
datasets, meta = build_station_data(cfg, SID)
corrector, model = build_models(cfg, meta['dims'])
print('dims', meta['dims'])

In [ ]:
from torch.utils.data import DataLoader
b = next(iter(DataLoader(datasets['train'], batch_size=8)))
for k,v in b.items():
    print(k, tuple(v.shape) if hasattr(v,'shape') else v)
y_hat, aux = model(b)
print('y_hat', y_hat.shape, 'preds', aux['preds'].shape, 'gate', aux['gate'].shape)

In [ ]:
# 订正前后（可配对列，归一化空间）
import matplotlib.pyplot as plt
if aux['corr_paired'] is not None:
    raw = b['nwp_paired'][0,:,0].detach().numpy()
    corr = aux['corr_paired'][0,:,0].detach().numpy()
    lmd = b['lmd_paired'][0,:,0].detach().numpy()
    plt.figure(figsize=(10,3))
    plt.plot(raw,label='nwp raw'); plt.plot(corr,label='corrected'); plt.plot(lmd,label='lmd true')
    plt.legend(); plt.title('correction (sample 0, paired col 0)'); plt.show()

In [ ]:
# 门控权重热图（batch × 专家）
g = aux['gate'].detach().numpy()
plt.figure(figsize=(6,4)); plt.imshow(g, aspect='auto'); plt.colorbar()
plt.xlabel('expert'); plt.ylabel('sample'); plt.title('gate weights'); plt.show()